# Build Snowflake Feature Store with getML

## Prerequisites

```bash
mise env --dotenv > notebooks/.env
```

In [1]:
import getml
from dotenv import load_dotenv

PROJECT_NAME = "snowflake_feature_store"

load_dotenv(dotenv_path=".env")
getml.set_project(PROJECT_NAME)

  Loading pipelines...                     ━━━━━━━━━━━━━━━ 100% • 00:00


Connected to project 'snowflake_feature_store'.

## Setup and Data Loading 

In [2]:
import os
from snowflake.snowpark import Session

connection_params: dict[str, str | int] = {
    "account": os.environ["SNOWFLAKE_ACCOUNT"],
    "user": os.environ["SNOWFLAKE_USER"],
    "password": os.environ["SNOWFLAKE_PASSWORD"],
    "role": os.environ["SNOWFLAKE_ROLE"],
    "warehouse": os.environ["SNOWFLAKE_WAREHOUSE"],
    "database": os.environ["SNOWFLAKE_DATABASE"],
    "schema": os.environ["SNOWFLAKE_SCHEMA"],
}

session = Session.builder.configs(connection_params).create()

In [3]:
weekly_sales_by_store = session.table(
    "PREPARED.WEEKLY_SALES_BY_STORE_WITH_TARGET"
).to_arrow()
weekly_sales_by_store = getml.DataFrame.from_arrow(
    weekly_sales_by_store, name="weekly_sales_by_store"
)

orders = session.table("RAW.RAW_ORDERS").to_arrow()
orders = getml.DataFrame.from_arrow(orders, name="orders")

DataFrame.to_arrow() is experimental since 1.28.0. Do not use it in production. 
/home/alex/projects/worktrees/getml-demo/60-create-initial-snowflake-notebook-5-sections/integration/snowflake/notebooks/.venv/lib/python3.12/site-packages/getml/data/_io/arrow.py:371: UserWarning:     
Column 'NEXT_WEEK_SALES' has been converted from decimal to float. This may
    result in a loss of precision!
    
  warnings.warn(


## getml Annotations

In [4]:
weekly_sales_by_store.set_role(
    cols=["STORE_ID", "SNAPSHOT_ID"], role=getml.data.roles.join_key
)
weekly_sales_by_store.set_role(cols="REFERENCE_DATE", role=getml.data.roles.time_stamp)
weekly_sales_by_store.set_role(cols="NEXT_WEEK_SALES", role=getml.data.roles.target)
weekly_sales_by_store.set_role(
    cols=[
        "STORE_NAME",
        "YEAR",
        "MONTH",
        "WEEK_NUMBER",
        "IS_FULL_WEEK_AFTER_OPENING",
        "HAS_ORDER_ACTIVITY",
        "HAS_MIN_HISTORY",
    ],
    role=getml.data.roles.categorical,
)
weekly_sales_by_store.set_role(
    cols=["DAYS_SINCE_OPEN", "NEXT_WEEK_ORDERS"], role=getml.data.roles.numerical
)


In [5]:
orders.set_role(cols=["STORE_ID", "ID", "CUSTOMER"], role=getml.data.roles.join_key)
orders.set_role(
    cols="ORDERED_AT",
    role=getml.data.roles.time_stamp,
    time_formats=["%Y-%m-%dT%H:%M:%S"],
)
orders.set_role(
    cols=["SUBTOTAL", "ORDER_TOTAL", "TAX_PAID"], role=getml.data.roles.numerical
)

In [6]:
weekly_sales_by_store

name,REFERENCE_DATE,STORE_ID,SNAPSHOT_ID,NEXT_WEEK_SALES,STORE_NAME,YEAR,MONTH,WEEK_NUMBER,IS_FULL_WEEK_AFTER_OPENING,HAS_ORDER_ACTIVITY,HAS_MIN_HISTORY,DAYS_SINCE_OPEN,NEXT_WEEK_ORDERS
role,time_stamp,join_key,join_key,target,categorical,categorical,categorical,categorical,categorical,categorical,categorical,numerical,numerical
unit,"time stamp, comparison only",,,,,,,,,,,,
0,2019-05-20,fc7707c0-2f1e-48d4-b870-7cbeddfc...,48,13165.49,Philadelphia,2019,5,21,true,true,true,261,1126
1,2020-04-27,eafbd328-0434-4f46-9c2d-cc97a46f...,145,24888.19,Brooklyn,2020,4,18,true,true,true,412,2346
2,2020-04-06,eafbd328-0434-4f46-9c2d-cc97a46f...,139,25648.41,Brooklyn,2020,4,15,true,true,true,391,2386
3,2019-07-08,eafbd328-0434-4f46-9c2d-cc97a46f...,61,10507.1,Brooklyn,2019,7,28,true,true,true,118,1113
4,2018-12-03,fc7707c0-2f1e-48d4-b870-7cbeddfc...,14,8689.4,Philadelphia,2018,12,49,true,true,true,93,737
,...,...,...,...,...,...,...,...,...,...,...,...,...
1374,2021-11-08,61743c9b-2394-4d36-8062-8a6820fa...,504,6451.9,Los Angeles,2021,11,45,true,true,true,57,550
1375,2022-04-18,c40b9cf5-6513-4f4c-a7fb-b72a1cdd...,645,15509.46,New Orleans,2022,4,16,true,true,true,405,1419


In [7]:
orders

name,ORDERED_AT,STORE_ID,ID,CUSTOMER,SUBTOTAL,ORDER_TOTAL,TAX_PAID
role,time_stamp,join_key,join_key,join_key,numerical,numerical,numerical
unit,"time stamp, comparison only",,,,,,
0,2020-05-18 07:44:00,fc7707c0-2f1e-48d4-b870-7cbeddfc...,1d78a778-9c83-4092-b185-3e0cb4c8...,2d54620c-28bd-44a4-aa0e-5031c01a...,600,636,36
1,2020-05-18 14:01:00,fc7707c0-2f1e-48d4-b870-7cbeddfc...,7d8ddf6a-c066-4121-a7fb-10572dcf...,0e0a7730-ba39-48ac-98fd-69293150...,400,424,24
2,2020-05-18 08:38:00,fc7707c0-2f1e-48d4-b870-7cbeddfc...,e664a36d-fcb6-4f6c-8473-b3264209...,d0ab73cc-8b20-4492-9bd9-ad922bea...,700,742,42
3,2020-05-18 14:43:00,fc7707c0-2f1e-48d4-b870-7cbeddfc...,cf5ee75a-68c4-45fe-872a-a7f5b005...,29e77d3b-14bf-4cd4-b4aa-54662ee2...,600,636,36
4,2020-05-18 17:00:00,fc7707c0-2f1e-48d4-b870-7cbeddfc...,0a73b088-99a4-424c-9ace-a9632fc5...,4c466e98-c2ea-438f-8be4-9669d9e1...,1600,1696,96
,...,...,...,...,...,...,...
2309598,2022-12-05 12:52:00,abfcc332-1eaf-42f6-b8e4-569bf6a3...,0bb3fbbf-bd42-4659-ae80-b28643cf...,be133bbe-beed-4e26-9844-bc742c3a...,400,425,25
2309599,2022-12-05 16:23:00,6964b061-b98d-43f2-9078-ef9c423e...,ea8bcf34-4d67-40e9-9568-3f20523c...,9b4479a5-0a03-4b2e-81b8-6e5df336...,2100,2257,157


## getML Data Model

In [8]:
validation_begin = getml.data.time.datetime(2023, 1, 1)
test_begin = getml.data.time.datetime(2024, 1, 1)

split = getml.data.split.time(
    population=weekly_sales_by_store,
    time_stamp="REFERENCE_DATE",
    validation=validation_begin,
    test=test_begin,
)

# Filter dataframes using the split column
weekly_sales_by_store_train = weekly_sales_by_store[split == "train"]
weekly_sales_by_store_validation = weekly_sales_by_store[split == "validation"]
weekly_sales_by_store_test = weekly_sales_by_store[split == "test"]

print(
    f"Training set size: {len(weekly_sales_by_store_train)}"
    f"\nValidation set size: {len(weekly_sales_by_store_validation)}"
    f"\nTest set size: {len(weekly_sales_by_store_test)}"
)

Training set size: 863
Validation set size: 312
Test set size: 204


In [9]:
weekly_sales_by_store_validation

name,REFERENCE_DATE,STORE_ID,SNAPSHOT_ID,NEXT_WEEK_SALES,STORE_NAME,YEAR,MONTH,WEEK_NUMBER,IS_FULL_WEEK_AFTER_OPENING,HAS_ORDER_ACTIVITY,HAS_MIN_HISTORY,DAYS_SINCE_OPEN,NEXT_WEEK_ORDERS
role,time_stamp,join_key,join_key,target,categorical,categorical,categorical,categorical,categorical,categorical,categorical,numerical,numerical
unit,"time stamp, comparison only",,,,,,,,,,,,
0,2023-02-27,abfcc332-1eaf-42f6-b8e4-569bf6a3...,914,24980.54,Chicago,2023,2,9,true,true,true,1035,2204
1,2023-05-15,fc7707c0-2f1e-48d4-b870-7cbeddfc...,983,18133.36,Philadelphia,2023,5,20,true,true,true,1717,1545
2,2023-05-22,abfcc332-1eaf-42f6-b8e4-569bf6a3...,986,23574.09,Chicago,2023,5,21,true,true,true,1119,2105
3,2023-08-28,fc7707c0-2f1e-48d4-b870-7cbeddfc...,1073,14535.01,Philadelphia,2023,8,35,true,true,true,1822,1260
4,2023-10-23,abfcc332-1eaf-42f6-b8e4-569bf6a3...,1118,26280.78,Chicago,2023,10,43,true,true,true,1273,2251
...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [10]:
data_model = getml.data.DataModel(
    population=weekly_sales_by_store_train.to_placeholder("WEEKLY_SALES_BY_STORE")
)

# Add all peripheral tables
data_model.add(
    getml.data.to_placeholder(
        orders=orders,
    )
)

# Define relationships using joins
data_model.WEEKLY_SALES_BY_STORE.join(
    right=data_model.orders,
    on="STORE_ID",
    time_stamps=("REFERENCE_DATE", "ORDERED_AT"),
    relationship=getml.data.relationship.one_to_many,
    memory=getml.data.time.days(30),
)


In [11]:
container = getml.data.Container(
    train=weekly_sales_by_store_train,
    validation=weekly_sales_by_store_validation,
    test=weekly_sales_by_store_test,
)

# Add peripheral tables with aliases matching the data model placeholders
container.add(
    orders=orders,
)
container.save()
getml.project.data_frames.save()

## Training

In [12]:
fast_prop = getml.feature_learning.FastProp()

predictor = getml.predictors.XGBoostRegressor(
    n_jobs=0,
)

pipe = getml.Pipeline(
    data_model=data_model,
    feature_learners=[
        fast_prop,
    ],
    predictors=[predictor],
)

pipe.fit(container.train)

Checking data model...

  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:00
  Checking...                              ━━━━━━━━━━━━━━━ 100% • 00:01


The pipeline check generated 0 issues labeled INFO and 2 issues labeled WARNING.

To see the issues in full, run .check() on the pipeline.

  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:01
  FastProp: Trying 50 features...          ━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features...           ━━━━━━━━━━━━━━━ 100% • 00:04
  XGBoost: Training as predictor...        ━━━━━━━━━━━━━━━ 100% • 00:01


Trained pipeline.

Time taken: 0:00:07.826460.



Pipeline(data_model='WEEKLY_SALES_BY_STORE',
         feature_learners=['FastProp'],
         feature_selectors=[],
         include_categorical=False,
         loss_function='SquareLoss',
         peripheral=['orders'],
         predictors=['XGBoostRegressor'],
         preprocessors=[],
         share_selected_features=0.5,
         tags=['container-8LIp3q'])

In [13]:
predictions = pipe.predict(container.test)

# Calculate metrics
scores = pipe.score(container.test)
scores

  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing...                         ━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features...           ━━━━━━━━━━━━━━━ 100% • 00:03
  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing...                         ━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features...           ━━━━━━━━━━━━━━━ 100% • 00:03


,date time,set used,target,mae,rmse,rsquared
0,2025-12-17 23:45:09,train,NEXT_WEEK_SALES,249.1798,316.897,0.9973
1,2025-12-17 23:45:17,test,NEXT_WEEK_SALES,481.3215,629.0032,0.9856


## Generate getML Feature Interpretations

In [14]:
from pathlib import Path

from getml_interpretations import (
    DomainReport,
    generate_domain_report,
)

domain_report_path = Path("domain_report.md")

if domain_report_path.exists():
    domain_report = DomainReport.from_markdown(domain_report_path)
else:
    domain_report: DomainReport = await generate_domain_report(
        project_name=PROJECT_NAME,
        pipeline=pipe,
        container=container,
        model="gpt-5.2",
    )
    domain_report.to_markdown(domain_report_path)

In [15]:
from getml_interpretations import (
    ColumnDescriptionsReport,
    generate_column_descriptions_report,
)

column_descriptions_report_path = Path("column_descriptions_report.json")
jaffle_shop_annotations = Path("annotations.yml")
user_prompt = f"""
Please use the following annotations as source of truth for generating the
column descriptions report:

{jaffle_shop_annotations.read_text()}getml_interpretations.
"""

if column_descriptions_report_path.exists():
    column_descriptions_report = ColumnDescriptionsReport.from_json(
        column_descriptions_report_path
    )
else:
    column_descriptions_report: ColumnDescriptionsReport = (
        await generate_column_descriptions_report(
            project_name=PROJECT_NAME,
            pipeline=pipe,
            container=container,
            domain_report=domain_report,
            model="gpt-5.2",
            user_prompt=user_prompt,
        )
    )
    column_descriptions_report.to_json(column_descriptions_report_path)

In [16]:
column_descriptions_report.model_dump()

{'column_descriptions': {'weekly_sales_by_store': {'REFERENCE_DATE': {'name': 'REFERENCE_DATE',
    'description': 'Weekly anchor timestamp used to align historical order activity to a specific store-week record. In the feature engineering, only transactions with order time at or before this timestamp are eligible, and a rolling lookback window of ~30 days is applied (i.e., orders within the prior 30 days relative to this date). Serves as the temporal index for train/validation/test splits (train: 2018-09 to 2022-12; validation: 2023; test: 2024). [timestamp]',
    'description_confidence': <Confidence.HIGH: 'high'>},
   'STORE_ID': {'name': 'STORE_ID',
    'description': 'Unique identifier for a physical store/location (6 distinct values). Primary join key connecting the weekly store-level table to the transactional orders table (one-to-many). Enables per-store aggregation of recent order history and supports store-level segmentation. [categorical key]',
    'description_confidence': 

In [17]:
from getml_interpretations import (
    FeatureDescriptionsReport,
    generate_feature_descriptions_report,
)

feature_descriptions_report_path = Path("feature_descriptions_report.json")

user_prompt = """
The feature descriptions are limited to 256 characters.
Please ensure that each feature description does not exceed this limit.
Keep the descriptions concise and informative.
"""

if feature_descriptions_report_path.exists():
    feature_descriptions_report = FeatureDescriptionsReport.from_json(
        feature_descriptions_report_path
    )
else:
    feature_descriptions_report: FeatureDescriptionsReport = (
        await generate_feature_descriptions_report(
            project_name=PROJECT_NAME,
            pipeline=pipe,
            container=container,
            domain_report=domain_report,
            column_descriptions_report=column_descriptions_report,
            model="gpt-5.2",
            batch_size=20,
        )
    )
    feature_descriptions_report.to_json(feature_descriptions_report_path)

In [18]:
feature_descriptions_report.model_dump()

{'features': {'feature_1_1': {'name': 'feature_1_1',
   'index': 0,
   'target': 'NEXT_WEEK_SALES',
   'importance': 0.0016171673773850389,
   'correlation': -0.007932532921688147,
   'sql': 'DROP TABLE IF EXISTS "FEATURE_1_1";\n\nCREATE TABLE "FEATURE_1_1" AS\nSELECT AVG( t2."subtotal" ) AS "feature_1_1",\n       t1.rowid AS rownum\nFROM "WEEKLY_SALES_BY_STORE__STAGING_TABLE_1" t1\nINNER JOIN "ORDERS__STAGING_TABLE_2" t2\nON t1."store_id" = t2."store_id"\nWHERE t2."ordered_at" <= t1."reference_date"\nAND ( t2."ordered_at__30_000000_days" > t1."reference_date" OR t2."ordered_at__30_000000_days" IS NULL )\nGROUP BY t1.rowid;'},
  'feature_1_2': {'name': 'feature_1_2',
   'index': 1,
   'target': 'NEXT_WEEK_SALES',
   'importance': 1.2975600440084582e-05,
   'correlation': 0.2434426817924243,
   'sql': 'DROP TABLE IF EXISTS "FEATURE_1_2";\n\nCREATE TABLE "FEATURE_1_2" AS\nSELECT COUNT( DISTINCT t2."subtotal" ) AS "feature_1_2",\n       t1.rowid AS rownum\nFROM "WEEKLY_SALES_BY_STORE__STA

## Feature Export

In [19]:
import pandas as pd

# Transform features using getML pipeline
features_np = pipe.transform(
    population_table=weekly_sales_by_store,
    peripheral_tables=[orders],
)

# Create DataFrame with meaningful feature names from getML pipeline
features_df = pd.DataFrame(features_np, columns=pipe.features.names)

# Add join keys for Snowflake Feature Store entity linking
features_df["STORE_ID"] = weekly_sales_by_store["STORE_ID"].to_numpy()
features_df["SNAPSHOT_ID"] = weekly_sales_by_store["SNAPSHOT_ID"].to_numpy()

# Uppercase column names to match Snowflake's unquoted identifier convention
features_df.columns = features_df.columns.str.upper()

# Write features to Snowflake table
session.write_pandas(
    df=features_df,
    table_name="GETML_FEATURES",
    schema="GETML_FS",
    auto_create_table=True,
    overwrite=True,
)

  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing...                         ━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features...           ━━━━━━━━━━━━━━━ 100% • 00:06


In [20]:
from snowflake.ml.feature_store import CreationMode, Entity, FeatureStore, FeatureView

# Create Snowpark DataFrame from the table for Feature Store
features_snowpark_df = session.table("GETML_FS.GETML_FEATURES")

# Initialize Feature Store
snowflake_feature_store = FeatureStore(
    session=session,
    database=os.environ["SNOWFLAKE_DATABASE"],
    name="GETML_FS",
    default_warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

# Create Entity (required for FeatureView)
store_entity = Entity(name="STORE_SNAPSHOT", join_keys=["STORE_ID", "SNAPSHOT_ID"])
snowflake_feature_store.register_entity(store_entity)


/home/alex/projects/worktrees/getml-demo/60-create-initial-snowflake-notebook-5-sections/integration/snowflake/notebooks/.venv/lib/python3.12/site-packages/snowflake/ml/feature_store/feature_store.py:189: UserWarning: Entity STORE_SNAPSHOT already exists. Skip registration.
  return f(self, *args, **kargs)


Entity(name=STORE_SNAPSHOT, join_keys=['STORE_ID', 'SNAPSHOT_ID'], owner=None, desc=)

In [21]:
feature_dict = feature_descriptions_report.model_dump()
# Build feature descriptions with uppercase keys to match Snowflake column names
# Extract description strings from nested dicts and uppercase the keys
feature_descs = feature_dict.get("feature_descriptions", {})
feature_descs_upper = {
    k.upper(): v.get("description", "") if isinstance(v, dict) else v
    for k, v in feature_descs.items()
}


# Create external FeatureView (refresh_freq=None means externally managed)
weekly_sales_feature_view = FeatureView(
    name="weekly_sales_features",
    entities=[store_entity],
    feature_df=features_snowpark_df,
    refresh_freq=None,
    desc="Features generated by getML for weekly sales prediction",
).attach_feature_desc(feature_descs_upper)

# Register the FeatureView
registered_feature_view = snowflake_feature_store.register_feature_view(
    feature_view=weekly_sales_feature_view, version="1", overwrite=True
)

registered_feature_view

FeatureView(_name=WEEKLY_SALES_FEATURES, _entities=[Entity(name=STORE_SNAPSHOT, join_keys=['STORE_ID', 'SNAPSHOT_ID'], owner=None, desc=)], _feature_df=<snowflake.snowpark.dataframe.DataFrame object at 0x7967135fc980>, _timestamp_col=None, _desc=Features generated by getML for weekly sales prediction, _infer_schema_df=<snowflake.snowpark.dataframe.DataFrame object at 0x796713775220>, _query=SELECT  *  FROM GETML_FS.GETML_FEATURES, _version=1, _status=FeatureViewStatus.STATIC, _feature_desc=OrderedDict({'FEATURE_1_1': 'Average order subtotal [currency, pre-tax] for a store over the historical lookback window ending at the store-week reference timestamp.\n\n**How it’s computed (SQL logic)**\n- Join the weekly store-level table to the transactional `orders` table on `STORE_ID` (one-to-many).\n- Keep only transactions with `orders.ORDERED_AT <= weekly_sales_by_store.REFERENCE_DATE` (no future leakage relative to the reference week).\n- Apply a rolling lookback of ~30 days using the generat

https://app.snowflake.com/pgciadt/sm09519/#/features/database/JAFFLE_SHOP/store/GETML_FS/feature-view/WEEKLY_SALES_FEATURES/version/1/feature-view-details

These descriptions integrate with Snowsight Universal Search, making features discoverable by searching their description text— Snowflake Documentationsnowflakesearching "promotional discount" would surface FEATURE_1_2.

## Snowflake Metadata Enrichment

Enrich Snowflake tables with comprehensive metadata from getML:
- Table and column comments on source and feature tables
- Semantic feature titles prepended to descriptions
- Feature importance and correlation as column tags
- Full feature metadata (including SQL) in a companion VARIANT table

In [ ]:
def set_column_comments(
    session: Session,
    object_name: str,
    column_descriptions: dict[str, dict],
    object_type: str = "TABLE",
) -> None:
    """Set column comments on a Snowflake table or view.

    For TABLEs: Uses COMMENT ON COLUMN syntax
    For VIEWs: Uses ALTER VIEW ... MODIFY COLUMN ... COMMENT syntax
    """
    for col_name, col_info in column_descriptions.items():
        description = col_info.get("description", "")
        description = description.replace("'", "''")

        if object_type == "VIEW":
            sql = f"ALTER VIEW {object_name} MODIFY COLUMN {col_name} COMMENT '{description}'"
        else:
            sql = f"COMMENT ON COLUMN {object_name}.{col_name} IS '{description}'"
        session.sql(sql).collect()


def set_column_tags(
    session: Session,
    object_name: str,
    column_name: str,
    tags: dict[str, str],
    object_type: str = "TABLE",
) -> None:
    """Set multiple tags on a column.

    Args:
        session: Snowflake session
        object_name: Fully qualified table/view name
        column_name: Column to tag
        tags: Dict of tag_name -> tag_value
        object_type: "TABLE" or "VIEW" (default: "TABLE")
    """
    for tag_name, tag_value in tags.items():
        tag_value_escaped = str(tag_value).replace("'", "''")
        sql = f"ALTER {object_type} {object_name} MODIFY COLUMN {column_name} SET TAG {tag_name} = '{tag_value_escaped}'"
        try:
            session.sql(sql).collect()
        except Exception as e:
            print(
                f"Warning: Could not set tag {tag_name} on {object_name}.{column_name}: {e}"
            )

In [23]:
# Set table/view comments on source and feature tables
# Note: PREPARED.WEEKLY_SALES_BY_STORE_WITH_TARGET is a VIEW, not a TABLE
object_descriptions = {
    "RAW.RAW_ORDERS": (
        "TABLE",
        "Raw order transactions from Jaffle Shop. Peripheral table for getML weekly_sales_features pipeline. Contains transactional data with ~2.3M orders spanning 2018-2024.",
    ),
    "PREPARED.WEEKLY_SALES_BY_STORE_WITH_TARGET": (
        "VIEW",
        "Prepared weekly sales data by store with target variable. Population table for getML weekly_sales_features pipeline. Contains store-week aggregations with NEXT_WEEK_SALES as prediction target.",
    ),
    "GETML_FS.GETML_FEATURES": (
        "TABLE",
        "ML features generated by getML FastProp for weekly sales prediction. Contains 50 time-series features derived from 30-day rolling aggregations of order data.",
    ),
}

for object_name, (object_type, description) in object_descriptions.items():
    desc_escaped = description.replace("'", "''")
    session.sql(f"COMMENT ON {object_type} {object_name} IS '{desc_escaped}'").collect()
    print(f"Set comment on {object_type} {object_name}")

Set comment on TABLE RAW.RAW_ORDERS
Set comment on VIEW PREPARED.WEEKLY_SALES_BY_STORE_WITH_TARGET
Set comment on TABLE GETML_FS.GETML_FEATURES


In [ ]:
# Set column comments on source tables using column_descriptions_report
# Map getml DataFrame names to Snowflake objects (with type)
object_mapping = {
    "weekly_sales_by_store": ("PREPARED.WEEKLY_SALES_BY_STORE_WITH_TARGET", "VIEW"),
    "orders": ("RAW.RAW_ORDERS", "TABLE"),
}

column_descs = column_descriptions_report.model_dump().get("column_descriptions", {})

for getml_name, (snowflake_object, object_type) in object_mapping.items():
    if getml_name in column_descs:
        print(f"Setting column comments on {object_type} {snowflake_object}...")
        set_column_comments(
            session, snowflake_object, column_descs[getml_name], object_type
        )
        print(f"  Set {len(column_descs[getml_name])} column comments")

Setting column comments on VIEW PREPARED.WEEKLY_SALES_BY_STORE_WITH_TARGET...
  Set 13 column comments
Setting column comments on TABLE RAW.RAW_ORDERS...
  Set 7 column comments


In [25]:
# Create column-level tags for feature metrics
feature_tags_sql = [
    "CREATE TAG IF NOT EXISTS GETML_FS.GETML_TITLE COMMENT = 'Semantic feature name from getML'",
    "CREATE TAG IF NOT EXISTS GETML_FS.GETML_IMPORTANCE COMMENT = 'XGBoost feature importance score'",
    "CREATE TAG IF NOT EXISTS GETML_FS.GETML_CORRELATION COMMENT = 'Pearson correlation with target variable'",
    "CREATE TAG IF NOT EXISTS GETML_FS.GETML_TARGET COMMENT = 'Target variable this feature predicts'",
]

for sql in feature_tags_sql:
    try:
        session.sql(sql).collect()
        print(f"Created tag: {sql.split('GETML_FS.')[1].split()[0]}")
    except Exception as e:
        print(f"Tag may already exist: {e}")

Created tag: GETML_TITLE
Created tag: GETML_IMPORTANCE
Created tag: GETML_CORRELATION
Created tag: GETML_TARGET


In [26]:
# Set enriched column comments and tags on feature table
# Merge features and feature_descriptions data
report = feature_descriptions_report.model_dump()
features_data = report.get("features", {})
feature_descs = report.get("feature_descriptions", {})

print(f"Processing {len(feature_descs)} features...")

# Build enriched column descriptions with title prefix
for feature_name, desc_info in feature_descs.items():
    col_name = feature_name.upper()
    title = desc_info.get("title", feature_name)
    description = desc_info.get("description", "")

    # Get importance/correlation from features dict
    feat_data = features_data.get(feature_name, {})
    importance = feat_data.get("importance", 0)
    correlation = feat_data.get("correlation", 0)

    # Build enriched description: "**title**: description [importance: X, correlation: Y]"
    enriched_desc = f"**{title}**: {description}\n\n[importance: {importance:.6f}, correlation: {correlation:.4f}]"
    enriched_desc_escaped = enriched_desc.replace("'", "''")

    # Set column comment
    session.sql(
        f"COMMENT ON COLUMN GETML_FS.GETML_FEATURES.{col_name} IS '{enriched_desc_escaped}'"
    ).collect()

    # Set column tags for structured querying
    set_column_tags(
        session,
        "GETML_FS.GETML_FEATURES",
        col_name,
        {
            "GETML_FS.GETML_TITLE": title,
            "GETML_FS.GETML_IMPORTANCE": f"{importance:.6f}",
            "GETML_FS.GETML_CORRELATION": f"{correlation:.4f}",
            "GETML_FS.GETML_TARGET": feat_data.get("target", "NEXT_WEEK_SALES"),
        },
    )

# Add join key descriptions
join_key_descs = {
    "STORE_ID": "Store identifier. Join key linking to weekly_sales_by_store population table.",
    "SNAPSHOT_ID": "Snapshot identifier. Join key for row-level entity identification.",
}
for col, desc in join_key_descs.items():
    desc_escaped = desc.replace("'", "''")
    session.sql(
        f"COMMENT ON COLUMN GETML_FS.GETML_FEATURES.{col} IS '{desc_escaped}'"
    ).collect()

print(
    f"Set enriched comments and tags on {len(feature_descs)} feature columns + 2 join keys"
)

Processing 52 features...
Set enriched comments and tags on 52 feature columns + 2 join keys


In [27]:
import json

# Create metadata table to store full feature info including SQL
session.sql("""
    CREATE TABLE IF NOT EXISTS GETML_FS.FEATURE_METADATA (
        FEATURE_NAME VARCHAR PRIMARY KEY,
        TITLE VARCHAR,
        IMPORTANCE FLOAT,
        CORRELATION FLOAT,
        TARGET VARCHAR,
        DESCRIPTION TEXT,
        DESCRIPTION_CONFIDENCE VARCHAR,
        SQL_CODE TEXT,
        FULL_METADATA VARIANT,
        CREATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
    ) COMMENT = 'getML feature metadata including SQL provenance for GETML_FEATURES table'
""").collect()

print("Created GETML_FS.FEATURE_METADATA table")

# Build a DataFrame for bulk insert using write_pandas
metadata_rows = []
for feature_name in features_data.keys():
    feat = features_data.get(feature_name, {})
    desc = feature_descs.get(feature_name, {})

    full_metadata = {**feat, **desc}

    metadata_rows.append(
        {
            "FEATURE_NAME": feature_name.upper(),
            "TITLE": desc.get("title", feature_name),
            "IMPORTANCE": feat.get("importance", 0),
            "CORRELATION": feat.get("correlation", 0),
            "TARGET": feat.get("target", ""),
            "DESCRIPTION": desc.get("description", ""),
            "DESCRIPTION_CONFIDENCE": desc.get("description_confidence", ""),
            "SQL_CODE": feat.get("sql", ""),
            "FULL_METADATA": json.dumps(full_metadata),
        }
    )

# Convert to pandas and write to Snowflake
metadata_df = pd.DataFrame(metadata_rows)

# Truncate and reload for idempotent operation
session.sql("TRUNCATE TABLE GETML_FS.FEATURE_METADATA").collect()
session.write_pandas(
    df=metadata_df,
    table_name="FEATURE_METADATA",
    schema="GETML_FS",
    overwrite=False,
)

print(f"Inserted {len(metadata_rows)} feature metadata records with SQL provenance")

Created GETML_FS.FEATURE_METADATA table
Inserted 52 feature metadata records with SQL provenance


In [ ]:
# Create and apply table-level getML tags
table_tags_sql = [
    "CREATE TAG IF NOT EXISTS GETML_FS.GETML_PIPELINE COMMENT = 'Name of getML pipeline using this data'",
    # Note: ALLOWED_VALUES syntax has no '=' sign
    "CREATE TAG IF NOT EXISTS GETML_FS.GETML_ROLE ALLOWED_VALUES 'population', 'peripheral', 'features', 'metadata' COMMENT = 'Role of table in getML data model'",
]

for sql in table_tags_sql:
    try:
        session.sql(sql).collect()
        print(f"Created tag: {sql.split('GETML_FS.')[1].split()[0]}")
    except Exception as e:
        print(f"Tag may already exist: {e}")

# Apply tags to tables/views
# Note: PREPARED.WEEKLY_SALES_BY_STORE_WITH_TARGET is a VIEW, not a TABLE
tag_assignments = [
    ("TABLE", "RAW.RAW_ORDERS", "GETML_FS.GETML_PIPELINE", "weekly_sales_features"),
    ("TABLE", "RAW.RAW_ORDERS", "GETML_FS.GETML_ROLE", "peripheral"),
    (
        "VIEW",
        "PREPARED.WEEKLY_SALES_BY_STORE_WITH_TARGET",
        "GETML_FS.GETML_PIPELINE",
        "weekly_sales_features",
    ),
    (
        "VIEW",
        "PREPARED.WEEKLY_SALES_BY_STORE_WITH_TARGET",
        "GETML_FS.GETML_ROLE",
        "population",
    ),
    (
        "TABLE",
        "GETML_FS.GETML_FEATURES",
        "GETML_FS.GETML_PIPELINE",
        "weekly_sales_features",
    ),
    ("TABLE", "GETML_FS.GETML_FEATURES", "GETML_FS.GETML_ROLE", "features"),
    (
        "TABLE",
        "GETML_FS.FEATURE_METADATA",
        "GETML_FS.GETML_PIPELINE",
        "weekly_sales_features",
    ),
    ("TABLE", "GETML_FS.FEATURE_METADATA", "GETML_FS.GETML_ROLE", "metadata"),
]

for object_type, object_name, tag, value in tag_assignments:
    try:
        session.sql(
            f"ALTER {object_type} {object_name} SET TAG {tag} = '{value}'"
        ).collect()
        print(f"Set {tag.split('.')[-1]}='{value}' on {object_name}")
    except Exception as e:
        print(f"Warning: Could not set tag on {object_name}: {e}")

Created tag: GETML_PIPELINE
Created tag: GETML_ROLE
Set GETML_PIPELINE='weekly_sales_features' on RAW.RAW_ORDERS
Set GETML_ROLE='peripheral' on RAW.RAW_ORDERS
Set GETML_PIPELINE='weekly_sales_features' on PREPARED.WEEKLY_SALES_BY_STORE_WITH_TARGET
Set GETML_ROLE='population' on PREPARED.WEEKLY_SALES_BY_STORE_WITH_TARGET
Set GETML_PIPELINE='weekly_sales_features' on GETML_FS.GETML_FEATURES
Set GETML_ROLE='features' on GETML_FS.GETML_FEATURES
Set GETML_PIPELINE='weekly_sales_features' on GETML_FS.FEATURE_METADATA
Set GETML_ROLE='metadata' on GETML_FS.FEATURE_METADATA


In [ ]:
# Verify metadata was applied
# Note: RAW, PREPARED, GETML_FS are schemas in the current database (JAFFLE_SHOP)

print("=== Table/View Comments ===")
for obj in [
    ("RAW", "RAW_ORDERS"),
    ("PREPARED", "WEEKLY_SALES_BY_STORE_WITH_TARGET"),
    ("GETML_FS", "GETML_FEATURES"),
]:
    schema, table_name = obj
    result = session.sql(
        f"SELECT COMMENT FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = '{schema}' AND TABLE_NAME = '{table_name}'"
    ).collect()
    comment = result[0][0][:80] if result and result[0][0] else "No comment"
    print(f"{schema}.{table_name}: {comment}...")

print("\n=== Sample Feature Column Comments ===")
result = session.sql("DESCRIBE TABLE GETML_FS.GETML_FEATURES").collect()
for row in result[:3]:
    comment = row["comment"] if row["comment"] else "No comment"
    print(f"  {row['name']}: {comment[:100]}...")

print("\n=== Sample Feature Tags ===")
try:
    result = session.sql("""
        SELECT * FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS('GETML_FS.GETML_FEATURES', 'TABLE'))
        LIMIT 8
    """).collect()
    for row in result:
        print(
            f"  {row['COLUMN_NAME']}.{row['TAG_NAME'].split('.')[-1]}: {row['TAG_VALUE']}"
        )
except Exception as e:
    print(f"  Could not query tags: {e}")

print("\n=== Feature Metadata Table (Top 5 by Importance) ===")
session.sql(
    "SELECT FEATURE_NAME, TITLE, IMPORTANCE, CORRELATION FROM GETML_FS.FEATURE_METADATA ORDER BY IMPORTANCE DESC LIMIT 5"
).show()

print("\n=== Sample SQL Code from Metadata ===")
result = session.sql(
    "SELECT FEATURE_NAME, SQL_CODE FROM GETML_FS.FEATURE_METADATA WHERE IMPORTANCE > 0 ORDER BY IMPORTANCE DESC LIMIT 1"
).collect()
if result:
    print(f"Feature: {result[0]['FEATURE_NAME']}")
    print(f"SQL:\n{result[0]['SQL_CODE'][:500]}...")

=== Table/View Comments ===
RAW.RAW_ORDERS: Raw order transactions from Jaffle Shop. Peripheral table for getML weekly_sales...
PREPARED.WEEKLY_SALES_BY_STORE_WITH_TARGET: Prepared weekly sales data by store with target variable. Population table for g...
GETML_FS.GETML_FEATURES: ML features generated by getML FastProp for weekly sales prediction. Contains 50...

=== Sample Feature Column Comments ===
  FEATURE_1_1: **avg_subtotal_last_30d_by_store**: Average order subtotal [currency, pre-tax] for a store over the ...
  FEATURE_1_2: **distinct_subtotal_values_last_30d_by_store**: Count of distinct subtotal amounts observed [count] ...
  FEATURE_1_3: **duplicate_subtotal_count_last_30d_by_store**: Number of “non-unique” subtotal occurrences [count] ...

=== Sample Feature Tags ===
  FEATURE_1_12.GETML_CORRELATION: -0.0049
  FEATURE_1_1.GETML_CORRELATION: -0.0079
  FEATURE_1_4.GETML_CORRELATION: -0.0362
  FEATURE_1_16.GETML_CORRELATION: -0.0385
  FEATURE_1_28.GETML_CORRELATION: -0.0756
 